# Slides 8â€“10: Product feedback data quality demo in VS Code â€” participant copy

**Scenario:** Before indexing product feedback for a foundation model, land the feedback in Amazon S3, catalog it with AWS Glue, validate it with DQDL, route content that needs review, and track a quality score.

**Start here:** Run the first code cell and edit only the clearly marked **PARTICIPANT INPUTS** block. The notebook defaults to a local-only run, so it does not call AWS until you deliberately set `RUN_AWS = True`.

Before enabling the AWS path:

1. Set `AWS_PROFILE` to the AWS CLI or SSO profile you intend to use, or leave it `None` for the normal boto3 credential chain.
2. Set `AWS_REGION` to the region for the exercise.
3. Replace `S3_BUCKET` with a globally unique S3 bucket name that you own and can write to.
4. Replace `GLUE_ROLE_ARN` with an IAM role ARN trusted by Glue. The role needs access to the S3 prefix and Glue Data Quality permissions; your notebook identity also needs permission to use the role with `iam:PassRole`.
5. Keep `PUBLISH_CLOUDWATCH = False` unless you intentionally want to publish metrics and accept any related charges.

The AWS path uploads this small product-feedback CSV to S3, creates or uses a Glue database, crawls the data, creates a DQDL ruleset, and starts an evaluation. It can create billable AWS resources. Run the cleanup cell after the exercise. AWS credentials should come from an existing profile or SSO login; never enter access keys or secrets into the notebook.

| Slide | What the class sees | AWS connection |
| --- | --- | --- |
| 8 | Product feedback in S3, Glue crawler, DQDL rules, evaluation results | S3 API, Glue Data Catalog, Glue Data Quality |
| 9 | A Lambda-style handler routes negative feedback for review | Optional live Amazon Comprehend sentiment call |
| 10 | Dimension scores, weighted score, threshold and chart | Optional CloudWatch custom metrics, disabled by default |

**Why Lambda?** Glue checks table-level structure and values. A Lambda function is useful for event-driven, record-level application decisions such as routing a complaint to human review, quarantining missing text, or invoking Comprehend. This notebook runs the same handler locally; it does not deploy Lambda.

**Classroom timing:** 5 minutes setup; 6 minutes S3 and Glue; 6 minutes DQDL evaluation; 5 minutes Lambda review routing; 6 minutes scoring; 4 minutes discussion.


In [ ]:
from collections import Counter
from datetime import datetime, timezone
from io import StringIO
import csv
import re

# --------------------------- PARTICIPANT INPUTS ---------------------------
# Leave RUN_AWS=False for the local classroom path. Set it to True only after
# completing every AWS placeholder below and confirming the expected charges.
RUN_AWS = False

# Optional AWS CLI profile or SSO profile. Leave None to use the normal boto3
# credential chain, for example environment credentials or an attached role.
AWS_PROFILE = None

# AWS Region where the S3, Glue, Comprehend, and optional CloudWatch calls run.
AWS_REGION = "us-east-1"

# Keep False unless you intentionally want to publish numeric metrics to CloudWatch.
PUBLISH_CLOUDWATCH = False

# Replace with a globally unique bucket name that you own and can write to.
S3_BUCKET = "<REPLACE_WITH_YOUR_UNIQUE_S3_BUCKET_NAME>"

# Object key written inside S3_BUCKET. The Glue crawler uses the product-feedback prefix.
S3_KEY = "product-feedback/feedback.csv"

# Glue Data Catalog database to create or reuse for this exercise.
GLUE_DATABASE = "product_feedback_training"

# Replace with the ARN of an IAM role trusted by AWS Glue. The role needs access
# to the S3 prefix and Glue Data Quality permissions for the AWS path.
GLUE_ROLE_ARN = "<REPLACE_WITH_YOUR_GLUE_SERVICE_ROLE_ARN>"

# Optional crawler name. Leave empty to generate a timestamped name automatically.
CRAWLER_NAME = ""

# Classroom scoring inputs. The weights must add up to 1.0.
QUALITY_WEIGHTS = {"Completeness": 0.4, "Accuracy": 0.4, "Consistency": 0.2}
QUALITY_THRESHOLD = 85

session = None
if RUN_AWS:
    import boto3
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

feedback = [
    {"feedback_id": "F001", "customer_id": "C01", "rating": 5, "content": "The new search is excellent and fast."},
    {"feedback_id": "F002", "customer_id": "C02", "rating": 1, "content": "The app keeps failing and support is terrible."},
    {"feedback_id": "F003", "customer_id": "C03", "rating": 4, "content": "Useful results, although the filters are confusing."},
    {"feedback_id": "F003", "customer_id": "C04", "rating": 6, "content": "I like the features but the response is slow."},
    {"feedback_id": "F005", "customer_id": "", "rating": 3, "content": ""},
]
feedback

## Slide 8 â€” Automated validation with AWS Glue Data Quality

**Say:** â€œThe model cannot fix missing IDs, duplicate records or invalid ratings. We define measurable checks before data reaches the model.â€

This first cell implements four **local teaching checks**. It does not call Glue and does not parse DQDL. The optional AWS section later runs the corresponding DQDL rules against a real Glue Data Catalog table. The local score below is a pass percentage across four checks; do not confuse it with the service-reported `Score` returned by Glue.


In [ ]:
ids = [row["feedback_id"] for row in feedback]
checks = {
    "Feedback ID completeness": all(bool(r["feedback_id"].strip()) for r in feedback),
    "Feedback ID uniqueness": len(ids) == len(set(ids)),
    "Rating within 1 to 5": all(isinstance(r["rating"], int) and 1 <= r["rating"] <= 5 for r in feedback),
    "Content completeness": all(bool(r["content"].strip()) for r in feedback),
}
for name, passed in checks.items():
    print(f'{"PASS" if passed else "FAIL"}  {name}')
local_rule_score = 100 * sum(checks.values()) / len(checks)
print(f"Local rule score: {local_rule_score:.0f}/100")

print("\nIssues to review:")
duplicate_ids = {feedback_id for feedback_id, count in Counter(ids).items() if count > 1}
for row in feedback:
    issues = []
    if row["feedback_id"] in duplicate_ids:
        issues.append("duplicate feedback_id")
    if not isinstance(row["rating"], int) or not 1 <= row["rating"] <= 5:
        issues.append("rating outside 1-5")
    if not row["customer_id"].strip():
        issues.append("missing customer_id")
    if not row["content"].strip():
        issues.append("missing content")
    if issues:
        print(row["feedback_id"], "; ".join(issues))

**Ask:** â€œWould a passing score of 50 mean half the rows are usable?â€ Answer: no. This score counts *rules passed*, not *rows passed*. Inspect the failed rules and affected records.

**Actual Glue DQDL example:** The rules below correspond to the same fields in a Data Catalog table. `IsComplete` demands all rows have a value; `IsUnique` demands no duplicate IDs. The rating rule rejects values outside 1â€“5. A table must exist before an evaluation can run. For this demonstration, create a CSV with the same columns in S3 and crawl it into a Glue table, or point to an existing table with compatible columns. Cataloging text and numeric column types correctly matters.


In [ ]:
DQDL = """Rules = [
    IsComplete "feedback_id",
    IsUnique "feedback_id",
    ColumnValues "rating" between 1 and 5,
    IsComplete "content"
]"""
print(DQDL)


## AWS path â€” land product feedback in S3 and catalog it

**Steps 1â€“2:** After completing the participant inputs and setting `RUN_AWS = True`, this cell writes the classroom records as CSV to your Amazon S3 bucket. AWS Glue then creates or reuses the database and crawler, reads the CSV schema, and creates the Data Catalog table used by DQDL. The crawler role needs access to the S3 prefix. This notebook does not create a bucket for you: `S3_BUCKET` must already be a globally unique bucket that you own.

The local path skips all AWS calls. The AWS path can take a few minutes and may incur S3 and Glue charges.

In [ ]:
s3_uri = None
glue_table = None
if RUN_AWS:
    from botocore.exceptions import ClientError

    missing_inputs = []
    if not S3_BUCKET or S3_BUCKET.startswith("<"):
        missing_inputs.append("S3_BUCKET")
    if not GLUE_ROLE_ARN or GLUE_ROLE_ARN.startswith("<"):
        missing_inputs.append("GLUE_ROLE_ARN")
    if missing_inputs:
        raise ValueError("Complete these participant inputs before setting RUN_AWS=True: " + ", ".join(missing_inputs))

    CRAWLER_NAME = CRAWLER_NAME or "product-feedback-crawler-" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
    sts = session.client("sts")
    account_id = sts.get_caller_identity()["Account"]
    s3 = session.client("s3")
    s3.head_bucket(Bucket=S3_BUCKET)

    csv_buffer = StringIO()
    writer = csv.DictWriter(csv_buffer, fieldnames=["feedback_id", "customer_id", "rating", "content"])
    writer.writeheader()
    writer.writerows(feedback)
    s3.put_object(Bucket=S3_BUCKET, Key=S3_KEY, Body=csv_buffer.getvalue().encode("utf-8"))
    s3_uri = f"s3://{S3_BUCKET}/{S3_KEY}"
    print("Uploaded product feedback:", s3_uri)

    glue = session.client("glue")
    try:
        glue.create_database(DatabaseInput={"Name": GLUE_DATABASE})
    except glue.exceptions.AlreadyExistsException:
        pass
    glue.create_crawler(
        Name=CRAWLER_NAME,
        Role=GLUE_ROLE_ARN,
        DatabaseName=GLUE_DATABASE,
        Targets={"S3Targets": [{"Path": f"s3://{S3_BUCKET}/product-feedback/"}]},
        SchemaChangePolicy={"UpdateBehavior": "UPDATE_IN_DATABASE", "DeleteBehavior": "LOG"},
    )
    glue.start_crawler(Name=CRAWLER_NAME)
    print("Started Glue crawler:", CRAWLER_NAME)
    import time
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        crawler = glue.get_crawler(Name=CRAWLER_NAME)["Crawler"]
        state = crawler["State"]
        crawl_status = crawler.get("LastCrawl", {}).get("Status", "RUNNING")
        print("Crawler status:", state, crawl_status)
        if state == "READY" and crawl_status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(15)
    else:
        raise TimeoutError(f"Crawler is still running; inspect {CRAWLER_NAME} in the Glue console.")
    if crawl_status != "SUCCEEDED":
        raise RuntimeError(f"Glue crawler ended with {crawl_status}.")
    tables = glue.get_tables(DatabaseName=GLUE_DATABASE)["TableList"]
    if not tables:
        raise RuntimeError("The crawler completed but created no Glue table.")
    glue_table = tables[0]["Name"]
    print("Glue database/table:", GLUE_DATABASE, glue_table)
else:
    print("AWS disabled: S3 upload and Glue crawler were not run.")

### Steps 3â€“4 â€” create DQDL rules and run the evaluation

**Step 3: Create the DQDL ruleset.** DQDL (Data Quality Definition Language) describes the checks Glue Data Quality applies to the Glue Catalog table. This demo uses the `product_feedback` table created by the crawler:

```text
Rules = [
    IsComplete "feedback_id",
    IsUnique "feedback_id",
    ColumnValues "rating" between 1 and 5,
    IsComplete "content"
]
```

| DQDL rule | Column | What it checks | Expected result for this sample |
| --- | --- | --- | --- |
| `IsComplete "feedback_id"` | `feedback_id` | Every record has a non-null feedback ID. | Pass: all five records have an ID. |
| `IsUnique "feedback_id"` | `feedback_id` | No two records share the same feedback ID. | Fail: `F003` appears twice. |
| `ColumnValues "rating" between 1 and 5` | `rating` | Every rating is numeric and within the business range 1â€“5. | Fail: one record has rating `6`. |
| `IsComplete "content"` | `content` | Every record contains feedback text. | Pass for this Glue rule because the crawled CSV represents the empty value as a non-null field; the local check separately identifies the blank `F005` content. |

`IsComplete` checks whether a field is present, while `IsUnique` checks whether values are duplicated across records. `ColumnValues ... between 1 and 5` expresses a business rule for valid product ratings. DQDL evaluates the table in S3 through the Glue Catalog schema; it does not use the local Python `checks` dictionary.

**Step 4: Run the evaluation.** The next code cell creates a timestamped ruleset, starts a Glue Data Quality evaluation run, waits for a terminal status, and retrieves the result. Glue returns:

- A result for each rule, such as `PASS` or `FAIL`, with an evaluation message.
- A service `Score`, which summarizes the ruleset result. In this four-rule example, two failed rules produce a score of `0.5`, displayed later as `50/100`.
- Result IDs that can be retained for investigation before the temporary resources are cleaned up.

The Glue role needs Data Quality permissions, S3 read access, and the notebook principal needs permission to create/start/read the evaluation plus `iam:PassRole`. The evaluation can take several minutes. Custom CloudWatch metrics remain disabled unless `PUBLISH_CLOUDWATCH = True`. After class, run the cleanup cell to delete the generated ruleset, crawler, database/catalog table, and uploaded object.

In [ ]:
glue_result = None
ruleset_name = None
if RUN_AWS:
    import time
    if not glue_table:
        raise ValueError("Run the S3 upload and Glue crawler cell first so glue_table is available.")
    table_ref = {"DatabaseName": GLUE_DATABASE, "TableName": glue_table}
    glue.get_table(DatabaseName=GLUE_DATABASE, Name=glue_table)
    ruleset_name = "product_feedback_dq_" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
    glue.create_data_quality_ruleset(Name=ruleset_name, Ruleset=DQDL, TargetTable=table_ref)
    run_id = glue.start_data_quality_ruleset_evaluation_run(
        DataSource={"GlueTable": table_ref},
        Role=GLUE_ROLE_ARN,
        RulesetNames=[ruleset_name],
        AdditionalRunOptions={"CloudWatchMetricsEnabled": PUBLISH_CLOUDWATCH},
    )["RunId"]
    print("Started Glue evaluation:", run_id, "ruleset:", ruleset_name)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        run = glue.get_data_quality_ruleset_evaluation_run(RunId=run_id)
        status = run["Status"]
        print("Status:", status)
        if status in ("SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"):
            break
        time.sleep(15)
    else:
        raise TimeoutError(f"Evaluation is still running; inspect Glue run {run_id} in the console.")
    if status != "SUCCEEDED":
        raise RuntimeError(run.get("ErrorString", f"Glue run ended with {status}"))
    result_ids = run.get("ResultIds", [])
    if not result_ids:
        raise RuntimeError("Glue reported success but returned no quality result IDs.")
    glue_result = glue.get_data_quality_result(ResultId=result_ids[0])
    print("Glue service score:", glue_result.get("Score"))
    for rule in glue_result.get("RuleResults", []):
        print(rule.get("Result"), rule.get("EvaluatedRule"), rule.get("EvaluationMessage", ""))
else:
    print("AWS disabled: DQDL was displayed, but Glue evaluation was not run.")

## Slide 9 â€” Use Lambda for record-level review routing

**Say:** â€œGlue validates the batch; Lambda reacts to each record.â€ Lambda is useful here because it can be triggered by an S3 event, inspect one product-feedback record, call Amazon Comprehend, and route the result immediately to human review, quarantine, or the normal pipeline. It is an application decision, not a table-quality rule. The notebook runs the same handler locally and does not deploy a Lambda function.

**Ask:** â€œIf a customer writes â€˜The app is terrible,â€™ should we delete the message?â€ Usually no. Retain it and prioritize review or service recovery. Sentiment is not a content-safety classifier.

The local path uses a small keyword rule so the class can see the event and decision without an AWS call. The optional AWS path calls Amazon Comprehend and may incur charges.

In [ ]:
def lambda_handler(event, context=None, comprehend_client=None):
    text = event.get("content", "").strip()
    if not text:
        return {"valid": False, "route": "quarantine", "reason": "Missing content"}
    if comprehend_client is not None:
        response = comprehend_client.detect_sentiment(Text=text, LanguageCode="en")
        sentiment = response["Sentiment"]
        source = "Amazon Comprehend"
    else:
        # Local demonstration rule only. It is not equivalent to Comprehend.
        sentiment = "NEGATIVE" if re.search(r"\b(terrible|failing|broken|bad)\b", text.lower()) else "OTHER"
        source = "local keyword example"
    return {"valid": True, "route": "human_review" if sentiment == "NEGATIVE" else "normal",
            "sentiment": sentiment, "source": source}

comprehend = None
if RUN_AWS:
    comprehend = session.client("comprehend")
for row in feedback:
    result = lambda_handler({"content": row["content"]}, comprehend_client=comprehend)
    print(row["feedback_id"], result)


**Instructor explanation:** In production, an S3 notification, DynamoDB stream, or scheduled event could invoke a deployed Lambda function. `event` would carry either text or a reference to a stored object; do not put an entire large file in the event. Add robust exception handling, language selection, text-size checks, and a privacy review before using real customer text.


## Slide 10 â€” Measure and track quality

**Say:** â€œOne overall score gives a quick signal. The dimension scores show what failed and why.â€ The next score is a **classroom-defined weighted score**, separate from the Glue rule score. The displayed weights are an example, not an AWS default.

Dimensions for this demo: completeness = fraction of records with content and customer ID; accuracy = ratings inside 1â€“5; consistency = uniqueness of feedback ID. A row can contribute to more than one dimension. Weights add to 1.0.


In [ ]:
n = len(feedback)
dimension_scores = {
    "Completeness": 100 * sum(bool(r["content"].strip()) and bool(r["customer_id"].strip()) for r in feedback) / n,
    "Accuracy": 100 * sum(isinstance(r["rating"], int) and 1 <= r["rating"] <= 5 for r in feedback) / n,
    "Consistency": 100 * len(set(ids)) / n,
}
weights = QUALITY_WEIGHTS
assert abs(sum(weights.values()) - 1.0) < 1e-9, "QUALITY_WEIGHTS must add up to 1.0"
weighted_score = sum(dimension_scores[k] * weights[k] for k in weights)
threshold = QUALITY_THRESHOLD
print("Dimension scores:")
for name, score in dimension_scores.items():
    print(f"  {name}: {score:.1f}/100 (weight {weights[name]:.0%})")
print(f"Weighted score: {weighted_score:.1f}/100 | Threshold: {threshold} | Status: {'PASS' if weighted_score >= threshold else 'INVESTIGATE'}")
if glue_result:
    glue_score = glue_result["Score"] * 100 if glue_result["Score"] <= 1 else glue_result["Score"]
    print(f"Glue service score: {glue_score:.1f}/100")
else:
    print("Glue service score: unavailable; AWS evaluation was not run")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(dimension_scores.keys(), dimension_scores.values(), color=["#2563eb", "#0891b2", "#7c3aed"])
ax.axhline(threshold, color="#b91c1c", linestyle="--", label=f"Class threshold {threshold}")
ax.set_ylim(0, 105)
ax.set_ylabel("Score / 100")
ax.set_title("Feedback data quality by dimension")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


### Step 6 â€” measure and track quality locally; CloudWatch is optional

The chart and weighted score are calculated locally for the class. The final cell can optionally publish only numeric scores to CloudWatch, but it requires both `RUN_AWS = True` and `PUBLISH_CLOUDWATCH = True`. Leave the flag `False` to avoid sending metrics. CloudWatch custom metric ingestion can incur charges.

In [ ]:
if RUN_AWS and PUBLISH_CLOUDWATCH:
    cloudwatch = session.client("cloudwatch")
    metric_data = [
        {"MetricName": "QualityScore", "Dimensions": [{"Name": "Dimension", "Value": name}],
         "Value": float(score), "Unit": "Percent"}
        for name, score in {**dimension_scores, "Overall": weighted_score}.items()
    ]
    cloudwatch.put_metric_data(Namespace="Training/DataQualityDemo", MetricData=metric_data)
    print("Published", len(metric_data), "numeric quality metrics to CloudWatch in", AWS_REGION)
else:
    print("CloudWatch disabled: scores were calculated locally and no metrics were published.")

In [ ]:
## Cleanup resources created for this run
if RUN_AWS:
    cleanup_glue = session.client("glue")
    cleanup_s3 = session.client("s3")

    if ruleset_name:
        cleanup_glue.delete_data_quality_ruleset(Name=ruleset_name)
        print("Deleted Glue ruleset:", ruleset_name)
    if CRAWLER_NAME:
        cleanup_glue.delete_crawler(Name=CRAWLER_NAME)
        print("Deleted Glue crawler:", CRAWLER_NAME)
    cleanup_glue.delete_database(Name=GLUE_DATABASE)
    print("Deleted Glue database and catalog table:", GLUE_DATABASE)
    cleanup_s3.delete_object(Bucket=S3_BUCKET, Key=S3_KEY)
    print("Deleted uploaded S3 object:", f"s3://{S3_BUCKET}/{S3_KEY}")
else:
    print("Cleanup skipped: AWS was disabled.")

## Debrief and next step

1. Ask learners to fix the duplicate `feedback_id`, the invalid rating, the empty content, and the missing customer ID in the first cell. Rerun slide 8 and slide 10 cells. Which scores rise?
2. Change the **business weighting** to completeness 0.6, accuracy 0.3, consistency 0.1. Explain why the overall score changes even though the records have not.
3. Explain why the slide 9 decision should retain negative feedback and route it for human review.
4. In a production pipeline, add an alarm for sustained low quality and retain failed-rule detail for investigation. A weighted overall score must never conceal a critical privacy or safety failure.

**AWS references:** [Glue DQDL](https://docs.aws.amazon.com/glue/latest/dg/dqdl.html), [create ruleset](https://docs.aws.amazon.com/boto3/latest/reference/services/glue/client/create_data_quality_ruleset.html), [start evaluation](https://docs.aws.amazon.com/boto3/latest/reference/services/glue/client/start_data_quality_ruleset_evaluation_run.html), [get quality result](https://docs.aws.amazon.com/boto3/latest/reference/services/glue/client/get_data_quality_result.html), [Comprehend detect sentiment](https://docs.aws.amazon.com/boto3/latest/reference/services/comprehend/client/detect_sentiment.html), [CloudWatch PutMetricData](https://docs.aws.amazon.com/boto3/latest/reference/services/cloudwatch/client/put_metric_data.html).
